This notebook is contains the basic training and evaluation loop for fine tuning Whisper. 
- See whisper-dataset-creation.ipynb to create a dataset from raw audio files
- Performance metric functions are found after the main training cycle

-------------------

Create the Whisper processor
- whisper-base is common
- Sets device appropriately



In [1]:
import transformers
from transformers import WhisperProcessor
import torch
import pandas as pd

model_name = "openai/whisper-large-v3"
language = "english"
task = "transcribe"

processor = WhisperProcessor.from_pretrained(model_name, language=language, task=task)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hide low level errors from transformers
transformers.logging.set_verbosity_error()

Create Whisper model. 
- If loading from a fine-tuned checkpoint use state_dict_path variable
- create_whisper_model automatically freezes all parameters except for the LM layer 

In [2]:
from transformers import WhisperForConditionalGeneration

def create_whisper_model(model_name='openai/whisper-base', device='cuda', state_dict_path=None):
    model = WhisperForConditionalGeneration.from_pretrained(model_name)
    print(f'Loaded {model_name} on {device}')

    if state_dict_path:
        model.load_state_dict(torch.load(state_dict_path))
        print(f'Loaded state dict from {state_dict_path}')

    # Send to device
    model.to(device)

    for param in model.parameters():
        param.requires_grad = False

    # Except those in the last layer
    for param in model.proj_out.parameters():
        param.requires_grad = True

    # Verify which layers are trainable
    print("\nTrainable parameters after freezing:")
    trainable_params = 0
    frozen_params = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            trainable_params += param.numel()
            print(f"  - {name} (Trainable, shape: {param.shape})")
        else:
            frozen_params += param.numel()
            # print(f"  - {name} (Frozen)") # Uncomment to see all frozen params

    total_params = trainable_params + frozen_params
    print(f"\nTotal trainable parameters: {trainable_params}")
    print(f"Total frozen parameters: {frozen_params}")
    print(f"Total parameters: {total_params}")
    print(f"Ratio of trained params to total params: {trainable_params / total_params:.4f}")

    return model

model = create_whisper_model(model_name=model_name, device=device)

Loaded openai/whisper-large-v3 on cuda

Trainable parameters after freezing:
  - model.decoder.embed_tokens.weight (Trainable, shape: torch.Size([51866, 1280]))

Total trainable parameters: 66388480
Total frozen parameters: 1477102080
Total parameters: 1543490560
Ratio of trained params to total params: 0.0430


Define DataCollator class for training

In [3]:
from transformers import DataCollatorForSeq2Seq
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

# --- Data Collator ---
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels since they have to be of different lengths and need different padding methods.
        # "input_features" for Whisper-based models (vs. "input_values" for wav2vec...)
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, 
                                                     return_tensors="pt",
                                                     return_attention_mask=True)
        
        labels_batch = self.processor.tokenizer.pad(label_features,
                                                    padding='longest', 
                                                    return_tensors="pt")

        # Replace padding with -100 for loss to work correctly correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, padding=True)

Load an already created dataset. 
- sample_percentage can be used to downsample the dataset for quick testing

In [4]:
from datasets import load_from_disk


def prepare_dataset(prepared_dataset_path, sample_percentage=None, seed=555):
    prepared_datasets = load_from_disk(prepared_dataset_path)

    print("--- Full Prepared Dataset ---")
    print(prepared_datasets)

    if not sample_percentage:
        print('-- No downsampling')
        return prepared_datasets

    # Downsample at the appropriate percentage
    train_split = prepared_datasets["train"]
    sampled_train_split = train_split.train_test_split(train_size=sample_percentage, shuffle=True, seed=seed)['train']

    val_split = prepared_datasets["val"]
    sampled_val_split = val_split.train_test_split(train_size=sample_percentage, seed=seed)['train'] 

    test_split = prepared_datasets["test"]
    sampled_test_split = test_split.train_test_split(train_size=sample_percentage, seed=seed)['train'] 

    # Overwrite the original splits with the sampled splits
    prepared_datasets['train'] = sampled_train_split
    prepared_datasets['val'] = sampled_val_split
    prepared_datasets['test'] = sampled_test_split

    print(f"\n--- Sampled ({sample_percentage*100}%) Dataset ---")
    print(prepared_datasets)

    return prepared_datasets

# Load full prepared dataset
prepared_dataset_path = 'DALI-fixed-explicit-only-large'

prepared_datasets = prepare_dataset(prepared_dataset_path=prepared_dataset_path, 
                                    #sample_percentage=.2,
                                    seed=555)

--- Full Prepared Dataset ---
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 1518
    })
    val: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 332
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 383
    })
})
-- No downsampling


Training parameters and dataloaders
- Sets learning rate, batch sizes, number of epochs, optimizer and LR scheduler

In [8]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
import re 
import os

def remove_punctuation(s):
    s = re.sub(r'[^a-zA-Z0-9\s]', '', s)
    return s.lower()

# Batch sizes for the dataloaders. Powers of 2 recommended for efficient memory usage
train_batch_size = 4
eval_batch_size = 4

# Defined train and test DLs
train_dataloader = DataLoader(prepared_datasets["train"], shuffle=True, collate_fn=data_collator, batch_size=train_batch_size)
eval_dataloader = DataLoader(prepared_datasets["val"], collate_fn=data_collator, batch_size=eval_batch_size)

# Set number of training epochs
num_epochs = 20
total_steps = len(train_dataloader) * num_epochs
num_warmup_steps = int(0.06 * total_steps)

# Optim and LR scheduler

lrs = {'medium.en': 6.25e-6, 'large-v3': 4e-6} # suggested by whisper

learning_rate = lrs['large-v3'] 
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = get_scheduler(name="linear",
                          optimizer=optimizer,
                          num_warmup_steps=num_warmup_steps,
                          num_training_steps=total_steps)

# Forces the model.generate method to transcribe audio interpreted as english
# -- not needed when starting with medium.en
# forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")

# Directory for best model to be saved
best_path_dir = "./whisper-ft-exp"
best_state_dict_path_dir = os.path.join(best_path_dir, "state_dict.bin")

# Baseline MER (medium was under 0.4...)
best_mer = 1

# Patience limit
patience_limit = 3

print('--Training--')
print(f'Total steps: {total_steps}')
print(f'Number of warmup steps: {num_warmup_steps}')

--Training--
Total steps: 7600
Number of warmup steps: 456


Main training cycle
- Automatically creates and saves a dataframe of the best MER output
- Patience counter for early exit of training

In [ ]:
from tqdm import tqdm
import jiwer

# use for early stopping of training if no increase in MER is detected
patience = 0 

for epoch in range(num_epochs):
    # train loop
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model.train()
    train_loss = 0
    
    for batch in tqdm(train_dataloader, desc=f"(Epoch {epoch+1} / {num_epochs}) Training "):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss

        # Backwards pass
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_dataloader)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # eval loop
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(eval_dataloader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # for medium.en
            # generated_ids = model.generate(input_features=batch["input_features"], 
            #                         attention_mask=batch["attention_mask"], 
            #                         num_beams=3, 
            #                         length_penalty=.8,
            #                         early_stopping=True,
            #                         pad_token_id=processor.tokenizer.pad_token_id,
            #                         eos_token_id=processor.tokenizer.eos_token_id)                    
            
            # large-v3
            generated_ids = model.generate(input_features=batch["input_features"], 
                                    attention_mask=batch["attention_mask"], 
                                    task='translate',
                                    language='en',
                                    temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
                                    pad_token_id=processor.tokenizer.pad_token_id,
                                    eos_token_id=processor.tokenizer.eos_token_id)  
            
            # Decode predictions
            predictions = processor.batch_decode(generated_ids, skip_special_tokens=True)
            
            # Decode labels, replacing -100 with pad token
            labels = batch["labels"].clone()
            labels[labels == -100] = processor.tokenizer.pad_token_id
            labels_str = processor.batch_decode(labels, skip_special_tokens=True)

            all_predictions.extend(predictions)
            all_labels.extend(labels_str)

    # Compute WER and MER. Since the text field lengths are of varied size, MER is a better metric for correcteness
    all_predictions = [remove_punctuation(p) for p in all_predictions]
    
    wer = jiwer.wer(all_predictions, all_labels)
    mer = jiwer.mer(all_predictions, all_labels)

    print(f"Avg training loss: {avg_train_loss:.4f} | Eval. MER: {mer:.5f}, WER: {wer:.5f}")
    print()

    # Save the model if it has the best MER so far
    if mer < best_mer:
        patience = 0 # reset patience counter
        best_mer = mer

        print(f"(!) New best MER: {best_mer:.5f}. Saving model...")

        # Save state dict
        torch.save(model.state_dict(), best_state_dict_path_dir)
        print(f"Model saved to {os.path.abspath(best_state_dict_path_dir)}")
        print()

        ## Create df to analyse the outputs for a best output
        to_add = []

        for i in range(len(all_predictions)):
            pred = all_predictions[i]
            actual = all_labels[i]
            mer = jiwer.mer(pred, actual)
            wer = jiwer.wer(pred, actual)

            to_add.append([pred, actual, mer, wer])

        df_best = pd.DataFrame(to_add, columns=['predicted', 'actual', 'mer', 'wer'])
        df_best.to_csv('best-mer-outputs.csv', index=False)

    else: patience += 1

    if patience == patience_limit: 
        print(f'No increase in MER detected in {patience_limit} rounds, breaking')
        break
        
print("\n--- Training Complete ---")
print(f"Best MER achieved: {best_mer}")

## Load the best model
print('\n -- Loading best state dict')
model.load_state_dict(torch.load(best_state_dict_path_dir))

(Epoch 1 / 20) Training :   2%|▏         | 9/380 [00:10<07:07,  1.15s/it]

--------------------------

WER MER scores
- Baseline medium.en (beam search with 3 beams) -  0.44619 | 0.40063

For investigating the outputs of whisper
- Load model if needed

In [ ]:
from transformers import WhisperForConditionalGeneration
import pandas as pd
import jiwer
import torch
import re

#model = create_whisper_model(model_name, device)

Evaluation cycle only. 
- Automatically creates a dataframe of the outputs

In [7]:
from tqdm import tqdm 
import jiwer

# forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")
device = 'cuda' if torch.cuda.is_available() else 'gpu'
model.eval()

all_predictions = []
all_norm_preds = []
all_labels = []

if torch.cuda.is_available():
        torch.cuda.empty_cache()

with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Evaluating"):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Generate predictions. Note this is different than model.transcribe (which is used for untrained?)
        generated_ids = model.generate(input_features=batch["input_features"], 
                                    attention_mask=batch["attention_mask"], 
                                    num_beams=3, 
                                    #length_penalty=.8,
                                    early_stopping=True,
                                    pad_token_id=processor.tokenizer.pad_token_id,
                                    eos_token_id=processor.tokenizer.eos_token_id
                                    )              
        
        # Decode predictions
        predictions = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # Decode labels, replacing -100 with pad token
        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        labels_str = processor.batch_decode(labels, skip_special_tokens=True)

        all_predictions.extend(predictions)
        all_labels.extend(labels_str)

# Create df for easier inspection
lst = []

for i in range(len(all_predictions)):
    pred = all_predictions[i]
    pred_norm = remove_punctuation(pred)
    actual = all_labels[i]

    wer = jiwer.wer(pred_norm, actual)
    mer = jiwer.mer(pred_norm, actual)

    # print(f'Predicted - {pred}')
    # print(f'Actual    - {actual}')
    # print(f'Jiwer WER: {wer} | MER: {mer}')
    # print()

    lst.append([pred, pred_norm, actual, wer, mer])

df_wer = pd.DataFrame(lst, columns=['Prediction (raw)', 'Prediction (normalized)', 'Actual', 'WER', 'MER'])
df_wer.to_csv('dali-fixed-baseline.csv', index=False)

# Overall WER and MER scores
jwer = jiwer.wer(df_wer['Prediction (normalized)'].tolist(), df_wer['Actual'].tolist())
jmer = jiwer.mer(df_wer['Prediction (normalized)'].tolist(), df_wer['Actual'].tolist())

print(f'Overall Jiwer wer: {jwer:.5f} | mer {jmer:.5f}')

Evaluating: 100%|██████████| 5790/5790 [1:36:37<00:00,  1.00s/it]  


Overall Jiwer wer: 0.42450 | mer 0.38339


For investigating the outputs. 

In [ ]:
#df_wer = pd.read_csv('wer0-dataset-base-untrained.csv')

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', None)

df_wer = df_wer.sort_values(by='WER')

df_wer

## FT wer: 0.78865 | mer 0.56762
## RAW

,Prediction (raw),Prediction (normalized),Actual,WER,MER
106,this situation,this situation,this situation,0.000000,0.000000
110,we just want more,we just want more,we just want more,0.000000,0.000000
121,some girls are bigger than others,some girls are bigger than others,some girls are bigger than others,0.000000,0.000000
124,let the rain fall down,let the rain fall down,let the rain fall down,0.000000,0.000000
130,it was blown away,it was blown away,it was blown away,0.000000,0.000000
1904,and i hope you feel the same,and i hope you feel the same,and i hope you feel the same,0.000000,0.000000
2076,no one can deny,no one can deny,no one can deny,0.000000,0.000000
2079,holding on to petty things,holding on to petty things,holding on to petty things,0.000000,0.000000
1161,all our scars will disappear,all our scars will disappear,all our scars will disappear,0.000000,0.000000
1163,what do you want from me now,what do you want from me now,what do you want from me now,0.000000,0.000000


------------------------------

Test a model on an audio file. 
- Audio does not need to be preprocessed

In [ ]:
import torchaudio

def test_transcribe(audio_path):
    # Put in evaluation mode
    model.eval()

    # Load audio file
    print(f"Loading audio from: {audio_path}...")
    waveform, sample_rate = torchaudio.load(audio_path)

    # Resample if necessary (Whisper expects 16kHz)
    if sample_rate != 16000:
        print(f"Resampling audio from {sample_rate}Hz to 16kHz...")
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
        sample_rate = 16000 # Update sample rate after resampling

    # Ensure mono audio (Whisper expects single channel)
    if waveform.shape[0] > 1:
        print("Converting stereo audio to mono...")
        waveform = waveform.mean(dim=0, keepdim=True) # Average channels to mono

    # Convert to numpy array (required by feature_extractor for raw audio)
    audio_array = waveform.squeeze().numpy()

    # Extract features (Mel spectrogram)
    processed_audio = processor.feature_extractor(audio_array, 
                                                  sampling_rate=sample_rate, 
                                                  return_tensors="pt",
                                                  return_attention_mask=True,
                                                  )
 
    input_features = processed_audio.input_features.to(device)
    attention_mask = processed_audio.attention_mask.to(device)

    print("Generating transcription...")
    with torch.no_grad():
        generated_ids = model.generate(input_features=input_features, 
                                       attention_mask=attention_mask,
                                       num_beams=3, 
                                       length_penalty=.8,
                                       early_stopping=True,
                                       task='transcribe',
                                       language='en',
                                       pad_token_id=processor.tokenizer.pad_token_id,
                                       eos_token_id=processor.tokenizer.eos_token_id)   
        
    # Create the transcription
    transcription = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return transcription

In [ ]:
# Load and preprocess the audio file
audio_path = 'vocals.wav'
test_transcript = test_transcribe(audio_path)

print("\nTranscription:\n", test_transcript)